# Phase 2 — Grid extension vs. standalone solar (share_dispersion)

For every rural community in the study region we compare two ways to bring it electricity:
extending the medium-voltage (MV) grid, or installing a standalone solar home system (SHS).
Whichever is annually cheaper wins. The HH-weighted fraction of communities where the SHS wins,
per cluster, is `share_dispersion`.

Steps: parameters → per-municipality demand → SHS sizing → breakeven per community →
cluster aggregation → robustness check → sensitivity analysis.


In [1]:
import os
import pandas as pd

BASE = os.path.dirname(os.getcwd())            # repo root (notebook lives in analyse_GIS_phase2/)
OUT = "output"
os.makedirs(OUT, exist_ok=True)

# Cluster -> municipality membership (same mapping as analyse data ramp/demande.ipynb)
CLUSTERS = {
    "C1": ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    "C2": ["Bolpebra"],
    "C3": ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    "C4": ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza",
           "Porvenir", "Puerto_Rico", "San_Lorenzo", "San_Pedro",
           "Santa_Rosa_Pando", "Santos_Mercado", "Sena", "Villa_Nueva"],
    "C5": ["Cobija"],
}
MUNI_TO_CLUSTER = {m: c for c, munis in CLUSTERS.items() for m in munis}


## 1. Parameters

Cost data is from **Peña Balderrama et al. 2020, *Energy for Sustainable Development* 56**,
Table A.5 (grid extension) and Table A.7 (standalone SHS). FX and discount rate are fixed
model-wide conventions.


In [2]:
FX = 0.924          # 1 USD = 0.924 EUR — 2024 annual average (IRS/ECB)
I_RATE = 0.10        # model discount rate — Misc_indep.json

# Raw USD costs — Peña Balderrama et al. 2020, ESD 56
MV_USD = 9_000        # USD/km, MV 33kV extension — Table A.5
CONN_USD = 150        # USD/HH, last-mile connection — Table A.5
PV_USD_KW = 5_500     # USD/kW, standalone PV — Table A.7
BAT_USD_KWH = 600     # USD/kWh, battery — Table A.7
OM_RATE = 0.02        # %/yr O&M on overnight capex — Tables A.5/A.7
LIFE_MV, LIFE_PV, LIFE_BAT = 30, 15, 10   # years — Tables A.5/A.7

MV_EUR = MV_USD * FX
CONN_EUR = CONN_USD * FX
PV_EUR_KW = PV_USD_KW * FX
BAT_EUR_KWH = BAT_USD_KWH * FX

# SHS sizing assumptions — standard tropical off-grid design
SYS_EFF = 0.75       # PV -> usable AC, incl. battery/inverter/wiring losses
BATT_DAYS = 2        # days of autonomy
BATT_DOD = 0.80       # depth of discharge


def crf(r, n):
    """Capital recovery factor: annualizes an overnight cost over n years at rate r."""
    return r * (1 + r) ** n / ((1 + r) ** n - 1)


CRF_MV = crf(I_RATE, LIFE_MV)
CRF_PV = crf(I_RATE, LIFE_PV)
CRF_BAT = crf(I_RATE, LIFE_BAT)

pd.DataFrame([
    {"Parameter": "FX (USD->EUR)", "Value": FX, "Source": "2024 annual avg, IRS/ECB"},
    {"Parameter": "i_rate", "Value": I_RATE, "Source": "Misc_indep.json"},
    {"Parameter": "MV extension", "Value": f"{MV_USD} USD/km -> {MV_EUR:.0f} EUR/km", "Source": "Peña Balderrama 2020, Table A.5"},
    {"Parameter": "Connection", "Value": f"{CONN_USD} USD/HH -> {CONN_EUR:.1f} EUR/HH", "Source": "Peña Balderrama 2020, Table A.5"},
    {"Parameter": "PV capex", "Value": f"{PV_USD_KW} USD/kW -> {PV_EUR_KW:.0f} EUR/kW", "Source": "Peña Balderrama 2020, Table A.7"},
    {"Parameter": "Battery capex", "Value": f"{BAT_USD_KWH} USD/kWh -> {BAT_EUR_KWH:.0f} EUR/kWh", "Source": "Peña Balderrama 2020, Table A.7"},
    {"Parameter": "O&M rate", "Value": OM_RATE, "Source": "Peña Balderrama 2020, Tables A.5/A.7"},
    {"Parameter": "Lifetime MV/PV/Bat", "Value": f"{LIFE_MV}/{LIFE_PV}/{LIFE_BAT} y", "Source": "Peña Balderrama 2020, Tables A.5/A.7"},
    {"Parameter": "System efficiency", "Value": SYS_EFF, "Source": "off-grid SHS design assumption"},
    {"Parameter": "Battery autonomy", "Value": f"{BATT_DAYS} days", "Source": "off-grid SHS design assumption"},
    {"Parameter": "Depth of discharge", "Value": BATT_DOD, "Source": "off-grid SHS design assumption"},
    {"Parameter": "CRF_MV/PV/Bat", "Value": f"{CRF_MV:.5f}/{CRF_PV:.5f}/{CRF_BAT:.5f}", "Source": f"CRF(i={I_RATE}, n)"},
])


,Parameter,Value,Source
0,FX (USD->EUR),0.924,"2024 annual avg, IRS/ECB"
1,i_rate,0.1,Misc_indep.json
2,MV extension,9000 USD/km -> 8316 EUR/km,"Peña Balderrama 2020, Table A.5"
3,Connection,150 USD/HH -> 138.6 EUR/HH,"Peña Balderrama 2020, Table A.5"
4,PV capex,5500 USD/kW -> 5082 EUR/kW,"Peña Balderrama 2020, Table A.7"
5,Battery capex,600 USD/kWh -> 554 EUR/kWh,"Peña Balderrama 2020, Table A.7"
6,O&M rate,0.02,"Peña Balderrama 2020, Tables A.5/A.7"
7,Lifetime MV/PV/Bat,30/15/10 y,"Peña Balderrama 2020, Tables A.5/A.7"
8,System efficiency,0.75,off-grid SHS design assumption
9,Battery autonomy,2 days,off-grid SHS design assumption


## 2. Per-municipality demand

Annual electricity per household, `per_hh_kWh = sum(all sufficiency_* columns except
sufficiency_water_heating) / total_hh + ECS_APPOINT_KWH_HH`.

- **RAMP load curves**: `analyse data ramp/sufficiency/data ramp/<municipality>/load_curve_energy_service_full_year_Norte_Amazonia.csv`
  (power in W at 1-minute steps over a year → kWh = W·min / 60,000)
- **Total households**: `exctraction of data/output/CSV_final.csv`, Bolivia Census 2024 column
  `NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Total`

**`sufficiency_water_heating` is excluded** from the electric SHS sizing: dispersed/off-grid
households do not run electric water heating in reality (thermosiphon, direct solar, or no hot
water at all), so including it would over-size the standalone PV+battery system and understate
`share_dispersion`. Only the small electric-resistance fraction found in the "appoint réaliste"
sufficiency variant (`EnergyScope_BO_nord_amazonia`, DEC_HP_ELEC/TS_DEC_HP_ELEC f_max=0, DEC_SOLAR
backed by DEC_DIRECT_ELEC as host tech) is added back, via `ECS_APPOINT_KWH_HH` below.

The RAMP "sufficiency" scenario is uniform by design, so every municipality should land close to
the same per-HH value — used here as a sanity check, not just a calculation.


In [3]:
csv_final = pd.read_csv(
    os.path.join(BASE, "exctraction of data/output/CSV_final.csv"), encoding="utf-8-sig"
)
hh_col = [c for c in csv_final.columns if "2024" in c and "Total" in c and "VIVIENDA" in c.upper()][0]


def std_muni_name(depto, raw):
    """Match CSV_final's municipality names to the CLUSTERS keys.
    'Santa Rosa' exists in both Beni and Pando — disambiguate by department."""
    raw = str(raw).strip()
    if raw == "Santa Rosa":
        return "Santa_Rosa_Beni" if depto == "Beni" else "Santa_Rosa_Pando"
    return raw.replace(" ", "_")


muni_rows = csv_final[csv_final["MUNICIPIO/TIOC"].notna()]
total_hh = {
    std_muni_name(row["DEPARTAMENTO"], row["MUNICIPIO/TIOC"]): row[hh_col]
    for _, row in muni_rows.iterrows()
    if std_muni_name(row["DEPARTAMENTO"], row["MUNICIPIO/TIOC"]) in MUNI_TO_CLUSTER
}
assert set(total_hh) == set(MUNI_TO_CLUSTER), "missing total_hh for some municipalities"


In [4]:
RAMP_DIR = os.path.join(BASE, "analyse data ramp/sufficiency/data ramp")

# APPOINT_ELEC_SHARE: region-wide fraction of decentralised hot-water (ECS / HEAT_LOW_T_DECEN)
# service met by the electric resistance host tech (DEC_DIRECT_ELEC) in the "appoint réaliste"
# sufficiency variant of EnergyScope_BO_nord_amazonia (Data/2025/sufficiency/C{1-5}/Technologies.csv:
# DEC_HP_ELEC and TS_DEC_HP_ELEC f_max=0, so DEC_SOLAR is backed by DEC_DIRECT_ELEC as host tech).
# Source: sufficiency_phase2 run (2026-07-19/20), region-wide DEC_DIRECT_ELEC HEAT_LOW_T_DECEN
# production / total "Hot water" demand = 3.9468 / 71.6685 GWh = 0.04825.
APPOINT_ELEC_SHARE = 0.04825
ECS_APPOINT_KWH_HH = 851 * APPOINT_ELEC_SHARE  # kWh/HH/yr added back as electric appoint

per_hh_kwh = {}
no_ramp_file = []

for muni in MUNI_TO_CLUSTER:
    path = os.path.join(RAMP_DIR, muni, "load_curve_energy_service_full_year_Norte_Amazonia.csv")
    if not os.path.exists(path):
        no_ramp_file.append(muni)
        continue
    df = pd.read_csv(path)
    # sufficiency_water_heating excluded on purpose (see section 2 markdown): dispersed/off-grid
    # households do not run electric water heating in reality. Only ECS_APPOINT_KWH_HH (the small
    # electric-resistance fraction from étape A) is added back below, uniformly per household.
    suff_cols = [c for c in df.columns if c.startswith("sufficiency_") and c != "sufficiency_water_heating"]
    kwh_total = df[suff_cols].sum().sum() / 60_000.0
    per_hh_kwh[muni] = kwh_total / total_hh[muni] + ECS_APPOINT_KWH_HH

# A few municipalities (e.g. Guayaramerín) have no RAMP run -> use the
# total_hh-weighted average of their cluster peers instead.
for muni in no_ramp_file:
    cluster = MUNI_TO_CLUSTER[muni]
    peers = [p for p in CLUSTERS[cluster] if p in per_hh_kwh]
    per_hh_kwh[muni] = (sum(per_hh_kwh[p] * total_hh[p] for p in peers)
                        / sum(total_hh[p] for p in peers))
    print(f"{muni}: no RAMP file -> fallback to {cluster} average over {peers}")

demand = pd.DataFrame([
    {"Cluster": MUNI_TO_CLUSTER[m], "Municipio": m, "total_hh": total_hh[m],
     "per_hh_kWh": round(per_hh_kwh[m], 1)}
    for m in MUNI_TO_CLUSTER
]).sort_values(["Cluster", "Municipio"])

mean_kwh = demand["per_hh_kWh"].mean()
demand["pct_dev_from_mean"] = (demand["per_hh_kWh"] / mean_kwh - 1) * 100
print(f"mean per_hh_kWh = {mean_kwh:.1f}  (sufficiency scenario is uniform by design; "
      f"ECS_APPOINT_KWH_HH = {ECS_APPOINT_KWH_HH:.2f} kWh/HH/yr added to every municipality)")
demand


mean per_hh_kWh = 1168.9  (sufficiency scenario is uniform by design; ECS_APPOINT_KWH_HH = 41.06 kWh/HH/yr added to every municipality)


,Cluster,Municipio,total_hh,per_hh_kWh,pct_dev_from_mean
0,C1,Exaltación,1455.0,1173.8,0.416743
3,C1,Ixiamas,3306.0,1131.4,-3.210510
1,C1,Reyes,3417.0,1158.1,-0.926367
2,C1,Santa_Rosa_Beni,2755.0,1176.3,0.630614
4,C2,Bolpebra,802.0,1166.8,-0.182096
5,C3,Guayaramerín,10891.0,1175.3,0.545066
7,C3,Puerto_Gonzalo_Moreno,1965.0,1173.4,0.382524
6,C3,Riberalta,27442.0,1174.9,0.510846
8,C4,Bella_Flor,1235.0,1169.4,0.040330
9,C4,Filadelfia,2514.0,1164.3,-0.395967


In [5]:
flagged = demand[demand["pct_dev_from_mean"].abs() > 5]
if len(flagged):
    print("FLAGGED — more than 5% off the mean:")
    print(flagged)
else:
    print("All municipalities within 5% of the mean — sanity check passed.")


All municipalities within 5% of the mean — sanity check passed.


## 3. Standalone SHS sizing

Each household's PV + battery system is sized to cover its own municipality's demand:

- PV size: `kWh/yr / (CF × 8760 × η_system)`, with `η_system = 0.75`
- Battery size: `(2 days autonomy × daily kWh) / depth_of_discharge`
- Solar capacity factor (CF) per cluster: mean of the `PV` column in
  `analyse data ramp/sufficiency/output_energyscope/<cluster>/Time_series.csv` (EnergyScope's own solar profile)


**Note — no thermosiphon/solar-water-heater capex added to the breakeven.** ECS (hot water) is
served in this model by `DEC_SOLAR` backed by `DEC_DIRECT_ELEC` as host tech (see étape A) — the
real-world equivalent is a solar thermosiphon with a small electric-resistance backup. This
equipment is needed by households on **both sides** of the breakeven comparison (grid-connected
and dispersed/SHS alike), so its cost cancels out and is deliberately **not** added to either the
`extension` or `standalone` annual cost in `classify_communities`. Only the electric-resistance
kWh (`ECS_APPOINT_KWH_HH`, added to `per_hh_kwh` above) affects PV+battery sizing, since that
portion is genuinely electric.


In [6]:
cf_pv = {}
for cluster in CLUSTERS:
    ts = pd.read_csv(
        os.path.join(BASE, f"analyse data ramp/sufficiency/output_energyscope/{cluster}/Time_series.csv"),
        sep=";", index_col=0,
    )
    cf_pv[cluster] = ts["PV"].mean()


def size_shs(kwh_per_year, cf, crf_pv=CRF_PV, crf_bat=CRF_BAT):
    """Return (overnight capex, annualized cost) in EUR per household."""
    pv_kw = kwh_per_year / (cf * 8760 * SYS_EFF)
    bat_kwh = (BATT_DAYS * kwh_per_year / 365) / BATT_DOD
    pv_eur = pv_kw * PV_EUR_KW
    bat_eur = bat_kwh * BAT_EUR_KWH
    capex = pv_eur + bat_eur
    annual = crf_pv * pv_eur + crf_bat * bat_eur + OM_RATE * capex
    return capex, annual


shs_capex, shs_annual = {}, {}
for muni, kwh in per_hh_kwh.items():
    shs_capex[muni], shs_annual[muni] = size_shs(kwh, cf_pv[MUNI_TO_CLUSTER[muni]])

pd.DataFrame([
    {"Cluster": MUNI_TO_CLUSTER[m], "Municipio": m,
     "overnight_capex_EUR_HH": round(shs_capex[m], 0),
     "annual_EUR_HH": round(shs_annual[m], 2)}
    for m in MUNI_TO_CLUSTER
]).sort_values(["Cluster", "Municipio"])


,Cluster,Municipio,overnight_capex_EUR_HH,annual_EUR_HH
0,C1,Exaltación,9678.0,1605.30
3,C1,Ixiamas,9328.0,1547.35
1,C1,Reyes,9548.0,1583.83
2,C1,Santa_Rosa_Beni,9698.0,1608.72
4,C2,Bolpebra,9707.0,1608.97
5,C3,Guayaramerín,9664.0,1603.39
7,C3,Puerto_Gonzalo_Moreno,9648.0,1600.77
6,C3,Riberalta,9661.0,1602.89
8,C4,Bella_Flor,9940.0,1644.45
9,C4,Filadelfia,9896.0,1637.22


## 4. Breakeven per community

For each community, compare the annualized cost of extending the grid against the annualized SHS
cost. Community-level inputs (distance to nearest line, household count) come from
`analyse_GIS_phase2/output/task1_community_distances_lines.csv`, built upstream in
`gis_phase2_analysis.ipynb`.

$$\text{annual\_extension} = (CRF_{MV} + OM) \times MV_{EUR/km} \times \frac{dist}{HH} + (CRF_{MV} + OM) \times CONN_{EUR}$$

A community is **dispersed** (SHS wins) if its annual SHS cost is lower than this. The breakeven
distance `d*` is where the two costs are equal — it scales with household count.


In [7]:
task1 = pd.read_csv(
    os.path.join(BASE, "analyse_GIS_phase2/output/task1_community_distances_lines.csv"),
    encoding="utf-8-sig",
)


def classify_communities(communities, shs_annual_by_muni, mv_eur_per_km, conn_eur, crf_mv):
    rows = []
    for _, row in communities.iterrows():
        muni, dist, hh = row["Municipio_std"], row["distance_km"], row["HH_2024_scaled"]
        if hh <= 0:
            continue
        standalone = shs_annual_by_muni[muni]
        slope = (crf_mv + OM_RATE) * mv_eur_per_km
        extension = slope * dist / hh + (crf_mv + OM_RATE) * conn_eur
        d_star = max((standalone - (crf_mv + OM_RATE) * conn_eur) * hh / slope, 0.0)
        rows.append({
            "Cluster": row["Cluster"], "Comunidad": row["Comunidad"], "Municipio_std": muni,
            "HH": hh, "distance_km": dist, "nearest_line_type": row["nearest_line_type"],
            "annual_extension_EUR_HH": round(extension, 2),
            "annual_standalone_EUR_HH": round(standalone, 2),
            "dispersed": standalone < extension,
            "breakeven_distance_km": round(d_star, 3),
        })
    return pd.DataFrame(rows)


communities = classify_communities(task1, shs_annual, MV_EUR, CONN_EUR, CRF_MV)
print(f"{len(communities)} communities: "
      f"{(~communities['dispersed']).sum()} connectable, {communities['dispersed'].sum()} dispersed")


697 communities: 372 connectable, 325 dispersed

## 5. Aggregate per cluster

`share_dispersion` is the household-weighted fraction of dispersed communities per cluster.


In [8]:
def aggregate_clusters(communities, mv_eur=MV_EUR, conn_eur=CONN_EUR, weighting="No_tiene_2012"):
    rows = []
    for cluster in CLUSTERS:
        sub = communities[communities["Cluster"] == cluster]
        total_hh_c   = sub["HH"].sum()                              # C-only HH
        total_hh_abc = sum(total_hh[m] for m in CLUSTERS[cluster])  # A+B+C denominator (CSV_final)
        dispersed_hh = sub.loc[sub["dispersed"], "HH"].sum()
        connectable = ~sub["dispersed"]
        extension_capex = (
            (sub.loc[connectable, "distance_km"] * mv_eur).sum()
            + sub.loc[connectable, "HH"].sum() * conn_eur
        )
        rows.append({
            "Cluster": cluster,
            "share_dispersion": round(dispersed_hh / total_hh_abc, 4) if total_hh_abc else 0.0,
            "extension_capex_EUR": round(extension_capex, 0),
            "mean_breakeven_km": round(sub["breakeven_distance_km"].mean(), 2),
            "median_breakeven_km": round(sub["breakeven_distance_km"].median(), 2),
            "N_connectable": int(connectable.sum()),
            "N_dispersed": int(sub["dispersed"].sum()),
            "total_HH": round(total_hh_abc, 1),
            "weighting": weighting,
        })
    return pd.DataFrame(rows)


share_dispersion = aggregate_clusters(communities)
share_dispersion


,Cluster,share_dispersion,extension_capex_EUR,mean_breakeven_km,median_breakeven_km,N_connectable,N_dispersed,total_HH,weighting
0,C1,0.1062,3844395.0,22.04,15.44,63,101,10933.0,No_tiene_2012
1,C2,0.1536,419341.0,24.00,20.08,5,11,802.0,No_tiene_2012
2,C3,0.0123,5508500.0,38.73,18.99,107,53,40298.0,No_tiene_2012
3,C4,0.0780,10284641.0,19.07,13.01,186,157,16612.0,No_tiene_2012
4,C5,0.0001,80570.0,42.30,7.08,11,3,15564.0,No_tiene_2012


In [9]:
communities.to_csv(os.path.join(OUT, "community_breakeven_detail.csv"), index=False)
share_dispersion.to_csv(os.path.join(OUT, "share_dispersion_final.csv"), index=False)


## 6. Robustness — household weighting

`task1` weights communities by `No_tiene_2012` (households without electricity in the 2012 census).
As a robustness check, redo it weighting by `Hogares_2012` (total households in 2012), from
`analyse_GIS_phase2/data/comunidades_electricidad_2012.csv`.

Note: that file encodes one municipality as *"Tercera Sección - Santa Rosa"* (Depto = Beni), which
the name-extraction regex turns into `"Santa Rosa"` — renamed below to `"Santa_Rosa_Beni"` so it
joins correctly with `task1`.


In [10]:
com2012 = pd.read_csv(
    os.path.join(BASE, "analyse_GIS_phase2/data/comunidades_electricidad_2012.csv"),
    encoding="utf-8-sig", low_memory=False,
)
com2012["Municipio_std"] = com2012["Municipio"].str.extract(r"- (.+)$")[0].str.strip()
com2012["Municipio_std"] = com2012["Municipio_std"].fillna(com2012["Municipio"].str.strip())

fix = (com2012["Municipio_std"] == "Santa Rosa") & (com2012["Depto"] == "Beni")
com2012.loc[fix, "Municipio_std"] = "Santa_Rosa_Beni"
print(f"renamed {fix.sum()} rows: 'Santa Rosa' (Beni) -> 'Santa_Rosa_Beni'")

com2012["Comunidad_norm"] = com2012["Comunidad"].str.strip().str.upper()
task1_norm = task1.copy()
task1_norm["Comunidad_norm"] = task1_norm["Comunidad"].str.strip().str.upper()

joined = task1_norm.merge(
    com2012[["Municipio_std", "Comunidad_norm", "Hogares"]],
    on=["Municipio_std", "Comunidad_norm"], how="left",
)
print(f"joined: {joined['Hogares'].notna().sum()}/{len(joined)} communities matched to Hogares_2012")

# Redistribute each cluster's 2024 household total proportionally to Hogares_2012
matched = joined[joined["Hogares"].notna()].copy()
matched["Hogares"] = pd.to_numeric(matched["Hogares"], errors="coerce")
cluster_totals = task1.groupby("Cluster")["HH_2024_scaled"].sum()
for cluster in CLUSTERS:
    mask = matched["Cluster"] == cluster
    matched.loc[mask, "HH_2024_scaled"] = (
        matched.loc[mask, "Hogares"] / matched.loc[mask, "Hogares"].sum() * cluster_totals[cluster]
    )

communities_hogares = classify_communities(matched, shs_annual, MV_EUR, CONN_EUR, CRF_MV)
share_dispersion_hogares = aggregate_clusters(communities_hogares, weighting="Hogares_2012")

compare = share_dispersion[["Cluster", "share_dispersion"]].rename(columns={"share_dispersion": "No_tiene_2012"}).merge(
    share_dispersion_hogares[["Cluster", "share_dispersion"]].rename(columns={"share_dispersion": "Hogares_2012"}),
    on="Cluster",
)
compare["delta"] = (compare["Hogares_2012"] - compare["No_tiene_2012"]).round(4)
compare


renamed 29 rows: 'Santa Rosa' (Beni) -> 'Santa_Rosa_Beni'
joined: 525/737 communities matched to Hogares_2012


,Cluster,No_tiene_2012,Hogares_2012,delta
0,C1,0.1062,0.0756,-0.0306
1,C2,0.1536,0.1186,-0.0350
2,C3,0.0123,0.0080,-0.0043
3,C4,0.0780,0.0425,-0.0355
4,C5,0.0001,0.0001,0.0000


## 7. Sensitivity

How much does `share_dispersion` move if the discount rate, MV cost, or SHS cost are off?
Grid: `i_rate ∈ {6.4%, 10%, 12%}` × `MV_cost ∈ {-50%, base, +50%}` × `SHS_cost ∈ {-25%, base, +25%}`.


In [11]:
RATES = [0.064, 0.10, 0.12]
MV_MULTIPLIERS = {"MV-50%": 0.50, "MV base": 1.00, "MV+50%": 1.50}
SHS_MULTIPLIERS = {"SHS-25%": 0.75, "SHS base": 1.00, "SHS+25%": 1.25}

sensitivity = []
for rate in RATES:
    crf_mv_r, crf_pv_r, crf_bat_r = crf(rate, LIFE_MV), crf(rate, LIFE_PV), crf(rate, LIFE_BAT)
    shs_annual_r = {m: size_shs(kwh, cf_pv[MUNI_TO_CLUSTER[m]], crf_pv_r, crf_bat_r)[1]
                    for m, kwh in per_hh_kwh.items()}
    standalone = communities["Municipio_std"].map(shs_annual_r)

    for mv_label, mv_mult in MV_MULTIPLIERS.items():
        extension = ((crf_mv_r + OM_RATE) * MV_EUR * mv_mult * communities["distance_km"] / communities["HH"]
                     + (crf_mv_r + OM_RATE) * CONN_EUR)
        for shs_label, shs_mult in SHS_MULTIPLIERS.items():
            dispersed = standalone * shs_mult < extension
            for cluster in CLUSTERS:
                mask = communities["Cluster"] == cluster
                total_hh_c = communities.loc[mask, "HH"].sum()
                dispersed_hh = communities.loc[mask & dispersed, "HH"].sum()
                sensitivity.append({
                    "i_rate": rate, "MV_scenario": mv_label, "SHS_scenario": shs_label,
                    "Cluster": cluster,
                    "share_dispersion": round(dispersed_hh / total_hh_c, 4) if total_hh_c else 0.0,
                })

sensitivity = pd.DataFrame(sensitivity)
sensitivity.to_csv(os.path.join(OUT, "share_dispersion_sensitivity.csv"), index=False)

In [12]:
for cluster in CLUSTERS:
    sub = sensitivity[sensitivity["Cluster"] == cluster].copy()
    sub["scenario"] = sub["MV_scenario"] + " / " + sub["SHS_scenario"]
    pivot = sub.pivot(index="i_rate", columns="scenario", values="share_dispersion")
    column_order = [f"{mv} / {shs}" for mv in MV_MULTIPLIERS for shs in SHS_MULTIPLIERS]
    print(f"\n{cluster}")
    display(pivot[column_order].round(4))



C1


scenario,MV-50% / SHS-25%,MV-50% / SHS base,MV-50% / SHS+25%,MV base / SHS-25%,MV base / SHS base,MV base / SHS+25%,MV+50% / SHS-25%,MV+50% / SHS base,MV+50% / SHS+25%
i_rate,,,,,,,,,
0.064,0.3423,0.2747,0.1966,0.5023,0.3994,0.3872,0.5409,0.5232,0.4987
0.100,0.3738,0.2943,0.2227,0.5232,0.4769,0.3955,0.6067,0.5348,0.5023
0.120,0.3872,0.3180,0.2427,0.5232,0.4987,0.3994,0.6137,0.5409,0.5232



C2


scenario,MV-50% / SHS-25%,MV-50% / SHS base,MV-50% / SHS+25%,MV base / SHS-25%,MV base / SHS base,MV base / SHS+25%,MV+50% / SHS-25%,MV+50% / SHS base,MV+50% / SHS+25%
i_rate,,,,,,,,,
0.064,0.2712,0.098,0.098,0.4869,0.4869,0.3627,0.4869,0.4869,0.4869
0.100,0.3627,0.098,0.098,0.4869,0.4869,0.4869,0.4869,0.4869,0.4869
0.120,0.3627,0.098,0.098,0.4869,0.4869,0.4869,0.4869,0.4869,0.4869



C3


scenario,MV-50% / SHS-25%,MV-50% / SHS base,MV-50% / SHS+25%,MV base / SHS-25%,MV base / SHS base,MV base / SHS+25%,MV+50% / SHS-25%,MV+50% / SHS base,MV+50% / SHS+25%
i_rate,,,,,,,,,
0.064,0.0670,0.0435,0.0335,0.1310,0.1107,0.0951,0.1573,0.1474,0.1247
0.100,0.0812,0.0494,0.0335,0.1474,0.1205,0.0999,0.1623,0.1505,0.1310
0.120,0.0951,0.0600,0.0407,0.1474,0.1236,0.0999,0.1623,0.1505,0.1396



C4


scenario,MV-50% / SHS-25%,MV-50% / SHS base,MV-50% / SHS+25%,MV base / SHS-25%,MV base / SHS base,MV base / SHS+25%,MV+50% / SHS-25%,MV+50% / SHS base,MV+50% / SHS+25%
i_rate,,,,,,,,,
0.064,0.1592,0.1051,0.0692,0.3417,0.2683,0.2037,0.4532,0.3649,0.3310
0.100,0.1909,0.1245,0.0857,0.3649,0.3079,0.2258,0.4653,0.3888,0.3417
0.120,0.1950,0.1356,0.0981,0.3667,0.3310,0.2347,0.4868,0.3972,0.3572



C5


scenario,MV-50% / SHS-25%,MV-50% / SHS base,MV-50% / SHS+25%,MV base / SHS-25%,MV base / SHS base,MV base / SHS+25%,MV+50% / SHS-25%,MV+50% / SHS base,MV+50% / SHS+25%
i_rate,,,,,,,,,
0.064,0.002,0.002,0.002,0.004,0.004,0.002,0.0040,0.004,0.004
0.100,0.002,0.002,0.002,0.004,0.004,0.003,0.0040,0.004,0.004
0.120,0.002,0.002,0.002,0.004,0.004,0.004,0.0139,0.004,0.004


## 8. Source B — off-grid households (Motor, Panel, Otra)

Source B households are already electrified off-grid in 2012 (diesel generator, solar panel, or
other). They live in the **same communities** as Source C, so the distance and line infrastructure
are shared. Adding them to each community increases its household count, which lowers the per-HH
line cost and can flip previously-dispersed communities to connectable.

Sources:
- **2012 community counts**: `analyse_GIS_phase2/data/comunidades_electricidad_2012.csv`,
  columns `Motor`, `Panel`, `Otra` (capped at `Hog_Elec - Red_elec` to avoid double-counting)
- **2024 municipal totals**: `exctraction of data/output/CSV_final.csv`, 2024 Motor/Panel/Otra columns


In [13]:
# Home-system unit sizes — aligned with analyse data ramp/reality/home_systems.ipynb
# Source: ENDE BO-L1222 / Programa de Electrificación Rural
PANEL_W = 50                   # Wp per panel-HH (ENDE PEVD kit: >=50 Wp policristalino)
GEN_W = 500                    # W per generator-HH (assumption, no field data — sensitivity at 350 W)
BATT_KWH_PER_PANEL_HH = 0.123  # kWh per panel-HH (ENDE PEVD kit: 123 Wh lithium 12V)


In [14]:
# ── 2024 municipal B totals from CSV_final ────────────────────────────────────
motor_col = [c for c in csv_final.columns if "2024" in c and "Motor" in c and "VIVIENDA" in c.upper()][0]
panel_col = [c for c in csv_final.columns if "2024" in c and "Panel" in c][0]
otra_col  = [c for c in csv_final.columns if "2024" in c and "Otra"  in c][0]
print("Source B columns from CSV_final:")
print(f"  motor: '{motor_col}'")
print(f"  panel: '{panel_col}'")
print(f"  otra:  '{otra_col}'")

muni_b = {}   # muni -> {total, motor, panel, otra}
for _, row in csv_final[csv_final["MUNICIPIO/TIOC"].notna()].iterrows():
    name = std_muni_name(row["DEPARTAMENTO"], row["MUNICIPIO/TIOC"])
    if name in MUNI_TO_CLUSTER:
        muni_b[name] = {
            "total": row[motor_col] + row[panel_col] + row[otra_col],
            "motor": row[motor_col],
            "panel": row[panel_col],
            "otra":  row[otra_col],
        }

b_check = pd.DataFrame([
    {"Cluster": MUNI_TO_CLUSTER[m], "Municipio": m,
     "Motor_2024": muni_b[m]["motor"], "Panel_2024": muni_b[m]["panel"],
     "Otra_2024": muni_b[m]["otra"], "B_total_2024": muni_b[m]["total"]}
    for m in MUNI_TO_CLUSTER
]).sort_values(["Cluster", "Municipio"])
print(f"\nTotal B households across all municipalities: {b_check['B_total_2024'].sum():.0f}")
b_check


Source B columns from CSV_final:
  motor: 'NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Motor propio'
  panel: 'NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Panel solar'
  otra:  'NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Otra'

Total B households across all municipalities: 9325


,Cluster,Municipio,Motor_2024,Panel_2024,Otra_2024,B_total_2024
0,C1,Exaltación,203.0,664.0,23.0,890.0
3,C1,Ixiamas,337.0,353.0,103.0,793.0
1,C1,Reyes,96.0,213.0,44.0,353.0
2,C1,Santa_Rosa_Beni,198.0,405.0,17.0,620.0
4,C2,Bolpebra,160.0,216.0,17.0,393.0
5,C3,Guayaramerín,166.0,383.0,97.0,646.0
7,C3,Puerto_Gonzalo_Moreno,48.0,58.0,126.0,232.0
6,C3,Riberalta,422.0,692.0,282.0,1396.0
8,C4,Bella_Flor,67.0,160.0,88.0,315.0
9,C4,Filadelfia,334.0,240.0,34.0,608.0


In [15]:
# ── 2012 community-level B counts → scale to 2024 ────────────────────────────
# Reuse com2012 from section 6 (Municipio_std normalized, Santa_Rosa_Beni fixed).
study_b = com2012[com2012["Municipio_std"].isin(MUNI_TO_CLUSTER)].copy()

# B_2012 = min(Motor+Panel+Otra, Hog_Elec - Red_elec) to avoid double-counting
study_b["B_2012_raw"] = (study_b["Motor"].fillna(0)
                         + study_b["Panel"].fillna(0)
                         + study_b["Otra"].fillna(0))
study_b["B_cap"]  = (study_b["Hog_Elec"].fillna(0) - study_b["Red_elec"].fillna(0)).clip(lower=0)
study_b["B_2012"] = study_b[["B_2012_raw", "B_cap"]].min(axis=1)
study_b["Panel_2012"] = study_b["Panel"].fillna(0)
study_b["Motor_2012"] = study_b["Motor"].fillna(0)
study_b["Otra_2012"]  = study_b["Otra"].fillna(0)
study_b["Comunidad_norm"] = study_b["Comunidad"].str.strip().str.upper()

# Scale each community's 2012 count proportionally to the 2024 municipal total
# (same method used for Source C scaling to 2024). Panel/Motor/Otra scale to their
# own 2024 municipal totals; "Otra" is kept separate here and split 50/50 later.
for col_2012, b_key in [("B_2012", "total"), ("Panel_2012", "panel"),
                        ("Motor_2012", "motor"), ("Otra_2012", "otra")]:
    col_2024 = col_2012.replace("_2012", "_2024")
    denom = study_b.groupby("Municipio_std")[col_2012].transform("sum")
    b24   = study_b["Municipio_std"].map({m: muni_b[m][b_key] for m in muni_b})
    study_b[col_2024] = (study_b[col_2012] / denom.replace(0, float("nan")) * b24).fillna(0)

# Aggregate to (Municipio_std, Comunidad_norm) — removes any duplicate rows
b_comm = (
    study_b.groupby(["Municipio_std", "Comunidad_norm"])[["B_2024", "Panel_2024", "Motor_2024", "Otra_2024"]]
    .sum()
    .reset_index()
)
print(f"Communities with B data: {len(b_comm)} (out of {len(task1)} in task1)")


Communities with B data: 489 (out of 697 in task1)


In [16]:
# ── Assign B values onto task1 without merging (avoids row duplication) ───────
# Some community names repeat within a municipality (e.g. "San Pedro" in Bella_Flor).
# Indexing by (Municipio_std, Comunidad_norm) preserves the exact 697-row structure of task1.
BCOLS = ["B_2024", "Panel_2024", "Motor_2024", "Otra_2024"]
b_indexed = b_comm.set_index(["Municipio_std", "Comunidad_norm"])[BCOLS]

task1_bc = task1.copy()
task1_bc["Comunidad_norm"] = task1_bc["Comunidad"].str.strip().str.upper()

key_index = pd.MultiIndex.from_frame(task1_bc[["Municipio_std", "Comunidad_norm"]])
task1_bc[BCOLS] = b_indexed.reindex(key_index).fillna(0.0).values

matched_b = (task1_bc["B_2024"] > 0).sum()
print(f"Communities with B households matched: {matched_b}/{len(task1_bc)}")
print(f"Total B households assigned: {task1_bc['B_2024'].sum():.0f}")
print(f"  (municipal totals from CSV_final: {sum(muni_b[m]['total'] for m in muni_b):.0f})")
print(f"  Unassigned B = due to community-name mismatch between 2012 census and GIS layer")

# HH for breakeven = C (no-tiene scaled) + B
task1_bc["HH_2024_scaled"] = task1_bc["HH_2024_scaled"] + task1_bc["B_2024"]

# Rerun breakeven — larger HH → lower per-HH extension cost → some dispersed may flip
communities_bc = classify_communities(task1_bc, shs_annual, MV_EUR, CONN_EUR, CRF_MV)

# Attach Panel/Motor/Otra for TECH_HS estimation.
# classify_communities skips rows with HH <= 0 and returns results in the same order —
# so we can align by position with the non-zero-HH rows of task1_bc.
task1_bc_valid = task1_bc[task1_bc["HH_2024_scaled"] > 0].reset_index(drop=True)
communities_bc["Panel_2024"] = task1_bc_valid["Panel_2024"].values
communities_bc["Motor_2024"] = task1_bc_valid["Motor_2024"].values
communities_bc["Otra_2024"]  = task1_bc_valid["Otra_2024"].values

# Count flips: communities dispersed under C-only that became connectable under B+C
merged_flip = communities[["Comunidad", "Municipio_std", "dispersed"]].merge(
    communities_bc[["Comunidad", "Municipio_std", "dispersed"]],
    on=["Comunidad", "Municipio_std"], suffixes=("_C", "_BC")
)
flipped = (merged_flip["dispersed_C"] & ~merged_flip["dispersed_BC"]).sum()
print(f"Communities that flipped dispersed→connectable by adding B: {flipped}")

Communities with B households matched: 411/697
Total B households assigned: 7269
  (municipal totals from CSV_final: 9325)
  Unassigned B = due to community-name mismatch between 2012 census and GIS layer


Communities that flipped dispersed→connectable by adding B: 94


### Updated share_dispersion and extension capex (B+C)

Denominator is now the **full** cluster household count from CSV_final (Source A + B + C), so the
resulting share represents the fraction of ALL households that end up off-grid.


In [17]:
def aggregate_clusters_bc(comm_bc):
    # Denominator = total HH per cluster from CSV_final (A+B+C)
    rows = []
    for cluster in CLUSTERS:
        sub = comm_bc[comm_bc["Cluster"] == cluster]
        total_hh_csv  = sum(total_hh[m] for m in CLUSTERS[cluster])   # A+B+C denominator
        hh_dispersed  = sub.loc[sub["dispersed"], "HH"].sum()
        connectable   = ~sub["dispersed"]
        ext_line = (sub.loc[connectable, "distance_km"] * MV_EUR).sum()
        ext_conn = sub.loc[connectable, "HH"].sum() * CONN_EUR         # C+B connectable HH

        # Dispersed B: existing off-grid equipment fed to EnergyScope as brownfield f_min.
        # Unit sizes from ENDE (PANEL_W / GEN_W / BATT_KWH_PER_PANEL_HH), aligned with home_systems.ipynb.
        # "Otra" (unspecified off-grid source) split 50/50 between PV and diesel, same as home_systems.
        disp = sub["dispersed"]
        disp_panel = sub.loc[disp, "Panel_2024"].sum() + sub.loc[disp, "Otra_2024"].sum() / 2
        disp_motor = sub.loc[disp, "Motor_2024"].sum() + sub.loc[disp, "Otra_2024"].sum() / 2

        rows.append({
            "Cluster": cluster,
            "share_dispersion_C":    share_dispersion.loc[
                share_dispersion["Cluster"] == cluster, "share_dispersion"].values[0],
            "share_dispersion_BC":   round(hh_dispersed / total_hh_csv, 4) if total_hh_csv else 0.0,
            "extension_capex_C_EUR": share_dispersion.loc[
                share_dispersion["Cluster"] == cluster, "extension_capex_EUR"].values[0],
            "extension_capex_BC_EUR": round(ext_line + ext_conn, 0),
            "N_connectable_BC": int(connectable.sum()),
            "N_dispersed_BC":   int(sub["dispersed"].sum()),
            # TECH_HS brownfield capacity for EnergyScope f_min (W -> MW, kWh -> MWh)
            "f_min_PV_HS_MW":      round(disp_panel * PANEL_W / 1e6, 4),
            "f_min_HS_DIESEL_MW":  round(disp_motor * GEN_W / 1e6, 4),
            "f_min_BATT_HS_MWh":   round(disp_panel * BATT_KWH_PER_PANEL_HH / 1e3, 4),
        })
    return pd.DataFrame(rows)


summary_bc = aggregate_clusters_bc(communities_bc)
summary_bc["delta_share"] = (summary_bc["share_dispersion_BC"] - summary_bc["share_dispersion_C"]).round(4)
summary_bc["delta_capex_EUR"] = summary_bc["extension_capex_BC_EUR"] - summary_bc["extension_capex_C_EUR"]
summary_bc


,Cluster,share_dispersion_C,share_dispersion_BC,extension_capex_C_EUR,extension_capex_BC_EUR,N_connectable_BC,N_dispersed_BC,f_min_PV_HS_MW,f_min_HS_DIESEL_MW,f_min_BATT_HS_MWh,delta_share,delta_capex_EUR
0,C1,0.1062,0.1211,3844395.0,14510579.0,85,79,0.0192,0.1093,0.0473,0.0149,10666184.0
1,C2,0.1536,0.1118,419341.0,1923418.0,11,5,0.0007,0.0118,0.0016,-0.0418,1504077.0
2,C3,0.0123,0.0099,5508500.0,11801908.0,131,29,0.0049,0.0320,0.0120,-0.0024,6293408.0
3,C4,0.0780,0.0680,10284641.0,17188456.0,214,129,0.0042,0.0717,0.0102,-0.0100,6903815.0
4,C5,0.0001,0.0000,80570.0,157242.0,14,0,0.0000,0.0000,0.0000,-0.0001,76672.0


### Documentation — changement d'hypothèses & crédit de l'équipement existant (hors breakeven)

Deux vérifications gardées **en dehors** du breakeven, pour les limitations de la thèse :

1. **Avant/après** l'harmonisation des tailles unitaires avec `home_systems.ipynb`
   (panneau 80 W → 50 W, ajout batterie, `Otra` désormais splitée 50/50 PV/diesel).
2. **Valeur du kit existant de la Source B** en % du coût standalone qu'elle paierait sinon.
   Ce crédit n'est **volontairement pas** soustrait dans `classify_communities` — il est déjà porté par
   le `f_min` brownfield d'EnergyScope (PV_HS / BATT_HS). Affiché ici uniquement pour quantifier sa petitesse.


In [18]:
# ── (documentation) Impact of harmonizing home-system unit sizes with home_systems.ipynb ──
# OLD phase2 assumptions: panel 80 W, motor 500 W, no battery, "Otra" excluded
# NEW (this notebook):    panel 50 W (=0.625x), motor 500 W, battery 0.123 kWh/panel-HH, "Otra" split 50/50
OLD_PANEL_W, OLD_GEN_W = 80, 500
cmp_rows = []
for cluster in CLUSTERS:
    sub = communities_bc[communities_bc["Cluster"] == cluster]
    d = sub["dispersed"]
    panel_old = sub.loc[d, "Panel_2024"].sum()                     # no Otra
    motor_old = sub.loc[d, "Motor_2024"].sum()                     # no Otra
    panel_new = panel_old + sub.loc[d, "Otra_2024"].sum() / 2      # +Otra/2
    motor_new = motor_old + sub.loc[d, "Otra_2024"].sum() / 2      # +Otra/2
    cmp_rows.append({
        "Cluster": cluster,
        "PV_HS_MW_old(80W)":      round(panel_old * OLD_PANEL_W / 1e6, 4),
        "PV_HS_MW_new(50W+Otra)": round(panel_new * PANEL_W / 1e6, 4),
        "DIESEL_MW_old":          round(motor_old * OLD_GEN_W / 1e6, 4),
        "DIESEL_MW_new(+Otra)":   round(motor_new * GEN_W / 1e6, 4),
    })
fmin_cmp = pd.DataFrame(cmp_rows)
print("f_min before/after harmonization (50/80 = 0.625 unit effect; the rest is the added Otra/2):")
display(fmin_cmp)

# ── (documentation, NOT applied to the breakeven) Value of B's EXISTING kit vs the standalone cost ──
# Kept out of classify_communities on purpose: this credit is already carried by EnergyScope's
# brownfield f_min (PV_HS / BATT_HS). Shown here only to quantify how small it is, for the thesis limitations.
kit_pv_capex  = (PANEL_W / 1000) * PV_EUR_KW            # EUR/panel-HH, overnight PV
kit_bat_capex = BATT_KWH_PER_PANEL_HH * BAT_EUR_KWH     # EUR/panel-HH, overnight battery
kit_annual = (CRF_PV * kit_pv_capex + CRF_BAT * kit_bat_capex
              + OM_RATE * (kit_pv_capex + kit_bat_capex))   # EUR/yr/panel-HH
print(f"\nExisting ENDE kit = {PANEL_W} Wp PV + {BATT_KWH_PER_PANEL_HH*1000:.0f} Wh battery "
      f"= {kit_pv_capex:.0f} + {kit_bat_capex:.0f} = {kit_pv_capex + kit_bat_capex:.0f} EUR overnight "
      f"-> {kit_annual:.1f} EUR/yr annualized")
credit_rows = []
for cluster in CLUSTERS:
    shs_cluster = sum(shs_annual[m] for m in CLUSTERS[cluster]) / len(CLUSTERS[cluster])  # mean standalone EUR/yr/HH
    credit_rows.append({
        "Cluster": cluster,
        "standalone_EUR_yr_HH":   round(shs_cluster, 1),
        "existing_kit_EUR_yr_HH": round(kit_annual, 1),
        "credit_if_applied_%":    round(100 * kit_annual / shs_cluster, 2),
    })
credit_doc = pd.DataFrame(credit_rows)
print("If B's existing kit were credited against the full-sufficiency standalone cost, it would lower it by:")
display(credit_doc)


f_min before/after harmonization (50/80 = 0.625 unit effect; the rest is the added Otra/2):


,Cluster,PV_HS_MW_old(80W),PV_HS_MW_new(50W+Otra),DIESEL_MW_old,DIESEL_MW_new(+Otra)
0,C1,0.0292,0.0192,0.0997,0.1093
1,C2,0.0010,0.0007,0.0114,0.0118
2,C3,0.0071,0.0049,0.0280,0.0320
3,C4,0.0046,0.0042,0.0587,0.0717
4,C5,0.0000,0.0000,0.0000,0.0000



Existing ENDE kit = 50 Wp PV + 123 Wh battery = 254 + 68 = 322 EUR overnight -> 51.0 EUR/yr annualized
If B's existing kit were credited against the full-sufficiency standalone cost, it would lower it by:


,Cluster,standalone_EUR_yr_HH,existing_kit_EUR_yr_HH,credit_if_applied_%
0,C1,1586.3,51.0,3.21
1,C2,1609.0,51.0,3.17
2,C3,1602.4,51.0,3.18
3,C4,1646.6,51.0,3.09
4,C5,1589.2,51.0,3.21


In [19]:
print("NOTE — denominator change:")
print("  share_dispersion_C   denominator = C-only HH per cluster (No_tiene_2012 scaled)")
print("  share_dispersion_BC  denominator = total HH per cluster from CSV_final (A+B+C)")
print()
print("ASSUMPTION flags (home-system unit sizes, aligned with home_systems.ipynb):")
print(f"  Panel size   : {PANEL_W} W/HH (ENDE PEVD kit, >=50 Wp)")
print(f"  Generator    : {GEN_W} W/HH (assumption, no field data — sensitivity at 350 W)")
print(f"  Battery      : {BATT_KWH_PER_PANEL_HH} kWh/panel-HH (ENDE PEVD kit, 123 Wh lithium)")
print("  'Otra' households split 50/50 between PV and diesel.")
print("  All can be revised once survey data is available.")


NOTE — denominator change:
  share_dispersion_C   denominator = C-only HH per cluster (No_tiene_2012 scaled)
  share_dispersion_BC  denominator = total HH per cluster from CSV_final (A+B+C)

ASSUMPTION flags (home-system unit sizes, aligned with home_systems.ipynb):
  Panel size   : 50 W/HH (ENDE PEVD kit, >=50 Wp)
  Generator    : 500 W/HH (assumption, no field data — sensitivity at 350 W)
  Battery      : 0.123 kWh/panel-HH (ENDE PEVD kit, 123 Wh lithium)
  'Otra' households split 50/50 between PV and diesel.
  All can be revised once survey data is available.


In [20]:
summary_bc.to_csv("output/share_dispersion_final_BC.csv", index=False)
communities_bc.to_csv("output/community_breakeven_detail_BC.csv", index=False)
print("Saved: output/share_dispersion_final_BC.csv")
print("Saved: output/community_breakeven_detail_BC.csv")


Saved: output/share_dispersion_final_BC.csv
Saved: output/community_breakeven_detail_BC.csv


## Outputs

All saved to `analyse_GIS_phase2/output/`:

**Source C only (sections 1–7)**
- `share_dispersion_final.csv` — share_dispersion and breakeven stats per cluster
- `community_breakeven_detail.csv` — per-community classification (C households only)
- `share_dispersion_sensitivity.csv` — sensitivity grid

**Source B+C (section 8)**
- `share_dispersion_final_BC.csv` — updated share_dispersion, extension capex, TECH_HS f_min
- `community_breakeven_detail_BC.csv` — per-community classification with C+B households
